In [ ]:
import pandas as pd
import numpy as np
import os
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, roc_curve)

from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis


In [53]:
df = pd.read_csv("C:/Users/PMLS/Desktop/ML_Classification_models/Data/student_mental_health_burnout_1M.csv")

In [54]:
df = df.dropna()

In [55]:
threshold = df['dropout_risk'].median()

df['dropout_risk'] = df['dropout_risk'].apply(
    lambda x: 1 if x > threshold else 0
)

In [56]:
print("Target Distribution:\n", df['dropout_risk'].value_counts())

Target Distribution:
 dropout_risk
1    500000
0    500000
Name: count, dtype: int64


In [57]:
X = df.drop(columns=['dropout_risk'])
y = df['dropout_risk']

In [58]:
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
numerical_cols = X.select_dtypes(include=['int64','float64']).columns.tolist()

C:\Users\PMLS\AppData\Local\Temp\ipykernel_29932\3106426004.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include=['object']).columns.tolist()


In [59]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [60]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ]
)

I used ColumnTransformer to apply StandardScaler to numerical features and OneHotEncoder to categorical features. This ensures that each type of data is processed appropriately before feeding it into machine learning models. I also used stratified train test split to maintain class balance.

In [62]:
import os
os.makedirs("models", exist_ok=True)

In [63]:
models = {
    "KNN": Pipeline([
        ('preprocessor', preprocessor),
        ('model', KNeighborsClassifier(n_neighbors=5))
    ]),
    
    "Decision Tree": Pipeline([
        ('preprocessor', preprocessor),
        ('model', DecisionTreeClassifier(random_state=42))
    ]),
    
    "Random Forest": Pipeline([
        ('preprocessor', preprocessor),
        ('model', RandomForestClassifier(n_estimators=100, random_state=42))
    ]),
    
    "Naive Bayes": Pipeline([
        ('preprocessor', preprocessor),
        ('model', GaussianNB())
    ]),
    
    "LDA": Pipeline([
        ('preprocessor', preprocessor),
        ('model', LinearDiscriminantAnalysis())
    ])
}

In [ ]:
results = {}
trained_models = {}

for name, pipeline in models.items():
    print(f"\n -> Training {name}...")
    
    # Train
    pipeline.fit(X_train, y_train)
    trained_models[name] = pipeline
    
    # Prediction
    y_pred = pipeline.predict(X_test)
    
    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    results[name] = acc
    
    print(f"{name} Results:")
    print(f"Accuracy: {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall: {rec:.4f}")
    print(f"F1 Score: {f1:.4f}")
    
    # Save model
    filename = f"models/{name.replace(' ', '_').lower()}.pkl"
    joblib.dump(pipeline, filename)
    
    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure()
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f"Confusion Matrix - {name}")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()


 -> Training KNN...


In [ ]:
names = list(results.keys())
values = list(results.values())

plt.figure(figsize=(8,5))
plt.bar(names, values)
plt.title("Model Comparison (Accuracy)")
plt.xlabel("Models")
plt.ylabel("Accuracy")
plt.xticks(rotation=30)
plt.show()

In [ ]:
best_model_name = max(results, key=results.get)
best_model = trained_models[best_model_name]

joblib.dump(best_model, "models/best_model.pkl")

print(" Best Model:", best_model_name)